# Phase 4: 合成データ・リカバリーテストを全7試合で実施 — Colab

**目的**: J03WPY 1試合で見つかった「二重メカニズム」仮説
(短い$\Delta t$の不安定性は測定誤差、長い$\Delta t$の不安定性はモデルの限界=真の時間依存性で説明できる、
`documents/phase4_synthetic_recovery_results.md`)が、他の試合でも再現するか確認する。

実行するロジックは `scripts/phase4_synthetic_recovery.py`(試合ごとにPINNを学習し、
Baseline1と同じ推定器を合成データ6条件[定数×ノイズ4段階、時間変化×ノイズ有無]に適用、
実データの$V_{max}(\Delta t)$曲線と比較)。

**実行前に**: メニューの `ランタイム > ランタイムのタイプを変更` で **GPU** を選択してください。

## 1. Google Drive をマウント(結果の永続化用)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/pi-fsm/phase4_synthetic_recovery_multi'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('results will be saved to', OUT_DIR)

## 2. リポジトリを取得

In [ ]:
%cd /content
if not os.path.exists('/content/pi-fsm'):
    !git clone https://github.com/ryu622/pi-fsm.git
%cd /content/pi-fsm
!git pull
!git log --oneline -5

## 3. 依存関係のインストール

In [ ]:
!pip install -q -e .

import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU detected — go to ランタイム > ランタイムのタイプを変更 > GPU')

## 4. 全7試合で合成データ・リカバリーテストを実行

試合ごとにPINNを学習(1試合あたりローカルCPU/MPSで約7〜8分、GPUならもっと速いはず)+ 6条件の合成シミュレーション。
中断しても完了済みの試合(`{OUT_DIR}/synthetic_recovery_<match_id>.csv` が存在するもの)はスキップされるので再実行して問題ない。

In [ ]:
!python scripts/phase4_synthetic_recovery.py --out-dir "{OUT_DIR}"

## 5. 結果の確認

In [ ]:
import pandas as pd
from IPython.display import Image, display

combined = pd.read_csv(f'{OUT_DIR}/synthetic_recovery_all_matches.csv')
print('試合ごとの条件別 Vmax(delta_t):')
display(combined.pivot_table(index=['match_id', 'delta_t'], columns='condition', values='vmax').round(2))

# 各試合について図を表示
for match_id in combined['match_id'].unique():
    print(f'--- {match_id} ---')
    display(Image(f'{OUT_DIR}/synthetic_recovery_{match_id}.png'))

## 次のステップ

- 各試合で「短い$\Delta t$は測定誤差(ノイズ条件のどれかに近い)、長い$\Delta t$は時間変化のみ・ノイズなし条件に近い」
  という同じパターンが見られれば、J03WPYの二重メカニズムの結論が一般的なものだと確認できる
- 結果はDriveの `{OUT_DIR}` に試合ごとのCSV/PNG + `synthetic_recovery_all_matches.csv` として永続化される
- 確認できたら `documents/phase4_synthetic_recovery_results.md` に7試合分の結果をまとめる